<table>
    <tr>
        <td>
            <h1>Etiquetage morpho-syntaxique</h1>
        </td>
    </tr>
</table>


<center><i>Réalisé par : </i>Douba JAFUNO </center>

<table style="width: 100%">
<tr>
    <td style="width: 15%">
    </td>
    <td style="width: 70%; text-align:left">
        <a href="#1"><h1>I. Introduction</h1></a><br>
           &nbsp; <a href="#presentation">I.1 Présentation du problème</a><br>
           &nbsp; <a href="#preparation">I.2 Préparation et Visualisation des données</a><br><br>

<a href="#2"><h1>II. Dictionnaires de Probabilité et de Tag (étiquette)</h1></a><br><br>
    &nbsp; <a href="#tc">II.1 Transition_counts</a><br>
    &nbsp; <a href="#ec">II.2 Emission_counts</a><br>
    &nbsp; <a href="#tac">II.3 Tag_counts</a><br>
    &nbsp; <a href="#tp">II.4 Test et Précision</a><br>
  

<a href="#3"><h1>III. Les chaînes de Markov</h1></a><br><br>
    &nbsp; <a href="#pré1">III.1 Présentation</a><br>
    &nbsp; <a href="#cmems">III.2 Chaîne de Markov et étiquetage morpho-syntaxique</a><br>
    &ensp; <a href="#gpt">III.2.1 Graphe et Probabilité de transition</a><br>
    &ensp; <a href="#mt">III.2.2 Matrice de Transition</a><br>
    &nbsp; <a href="#mmc">III.3 Modèle de Markov cachés</a><br>
    &ensp; <a href="#pré2">III.3.1 Présentation</a><br>
    &ensp; <a href="#pme">III.3.2 Probabilité et matrice d'émission</a><br>
    &ensp; <a href="#ecpt">III.3.3 Exemple calcul de probabilité de translation</a><br>
    &nbsp; <a href="#rp">III.4 Retour au Problème</a><br>
    &ensp; <a href="#cmta">III.4.1 Création de la matrice de transition A</a><br>
    &ensp; <a href="#cmeb">III.4.2 Création de la matrice d'émission B</a><br>

<a href="#4"><h1>IV. Algorithme de Viterbi</h1></a><br><br>
    &nbsp; <a href="#ini">IV.1 Initialisation</a><br>
    &nbsp; <a href="#vf">IV.2 Viterbi forward</a><br>
    &nbsp; <a href="#vb">IV.2 Viterbi backward</a><br>


<a href="#5"><h1>V. Référence </h1></a><br><br>
   </td>
    <td style="width: 0%">
    </td>
</tr>
</table>


# <a name="1">I. Introduction</a>

## <a name="presentation"> Présentation du problème </a>

**l'étiquetage morpho-syntaxique** (aussi appelé étiquetage grammatical, POS tagging (part-of-speech tagging) en anglais) est le processus qui consiste à associer aux mots d'un texte les informations grammaticales correspondantes comme le sujet de la phrase, le genre, le nombre etc. à l'aide d'un outil informatique il identifie les nom, verbes et celà et très utile pour la compréhension dans les phrases, on peut aussi l'utiliser en reconnaissance vocale, il vérifie si une séquence de mots à une probabilité élevé

Terme lexical   Etiquette (POS Tag)     exemple

noun                NN              something, nothing
verb                VB              learn, study
determiner          DT                 the, a
w-adverb            WRB               why, where

Cepandant certains mots peuvent avoir plusieur **Tag ou étiquettes** comme well et celà peut etre ambigue  

- The whole team played **well**. [adverb]  (bien)
- You are doing **well** for yourself. [adjective] (très bien)
- **Well**, this assignment took me forever to complete. [interjection] (Eh bien)
- The **well** is dry. [noun] (le bien)
- Tears were beginning to **well** in her eyes. [verb] (couler)

## <a name="preparation"> Préparation et Visualisation des données</a>




Dans cette partie on traitera deux ensembles de données etiquetée recueillies dans le **Wall Street Journal (WSJ)**.

[Ici](http://relearn.be/2015/training-common-sense/sources/software/pattern-2.6-critical-fork/docs/html/mbsp-tags.html) il s'agit d'un exemple de "tag-set" ou de désignation "Part of Speech" (POS) décrivant le tag (l'étiquette) à deux ou trois lettres et sa signification.
- Un ensemble de données (**WSJ-2_21.pos**) sera utilisé pour la **formation (train)**.
- L'autre (**WSJ-24.pos**) sera utilisé pour le **test**.
- Les données de formation marquées ont été prétraitées pour former un vocabulaire (**hmm_vocab.txt**).
- Les mots du vocabulaire sont des mots de l'ensemble de formation qui ont été utilisés deux fois ou plus.
- Le vocabulaire est enrichi d'un ensemble de "mots-clés inconnus", décrits ci-dessous.

L'ensemble d'apprentissage sera utilisé pour créer les décomptes d'émission, de transmission et d'étiquettes (tag).

L'ensemble de test (WSJ-24.pos) est lu pour créer "y".
- Il contient à la fois le texte de test et le vrai tag .
- L'ensemble de test a également été prétraité pour supprimer les étiquettesafin de former **test_words.txt**.
- Celui-ci est lu et traité pour identifier la fin des phrases et traiter les mots ne faisant pas partie du vocabulaire à l'aide des fonctions fournies dans la cellule ci dessus.
- Cela forme la liste `prep`, le texte prétraité utilisé pour tester nos POS tag.

Un POS tag  rencontrera nécessairement des mots qui ne sont pas dans ses jeux de données.
- Pour améliorer la précision, ces mots sont analysés plus en détail pendant le prétraitement (preprocessing) afin d'extraire les indices disponibles quant à leur balise appropriée.
- Par exemple, le suffixe "ize" est un indice que le mot est un verbe, comme dans "final-ize" ou "character-ize".
- Un ensemble de mots inconnus, tels que "--unk-verb--" ou "--unk-noun--", remplacera les mots inconnus dans le corpus de formation et de test et apparaîtra dans les structures de données d'émission, de transmission et de tag.

Le but de cette partie est de mettre en place un modèle qui prédit le Tag d'un mot

<img src = "images/DataSources1.PNG" />

In [ ]:
import string
import pandas as pd
from collections import defaultdict
import math
import numpy as np



punct = set(string.punctuation)

# Règles de morphologie utilisées pour attribuer des tokens de mots inconnus
noun_suffix = ["action", "age", "ance", "cy", "dom", "ee", "ence", "er", "hood", "ion", "ism", "ist", "ity", "ling", "ment", "ness", "or", "ry", "scape", "ship", "ty"]
verb_suffix = ["ate", "ify", "ise", "ize"]
adj_suffix = ["able", "ese", "ful", "i", "ian", "ible", "ic", "ish", "ive", "less", "ly", "ous"]
adv_suffix = ["ward", "wards", "wise"]

# Nous utiliserons les fonctions suivantes
def get_word_tag(line, vocab):
    if not line.split():
        word = "--n--"
        tag = "--s--"
        return word, tag
    else:
        word, tag = line.split()
        if word not in vocab:
            # Gérer les mots inconnus
            word = assign_unk(word)
        return word, tag
    return None


def preprocess(vocab, data_fp):
    """
    Preprocess data
    """
    orig = []
    prep = []

    # lire les données
    with open(data_fp, "r") as data_file:

        for cnt, word in enumerate(data_file):

            # fin de phrases
            if not word.split():
                orig.append(word.strip())
                word = "--n--"
                prep.append(word)
                continue

            # Gérer les mots inconnus
            elif word.strip() not in vocab:
                orig.append(word.strip())
                word = assign_unk(word)
                prep.append(word)
                continue

            else:
                orig.append(word.strip())
                prep.append(word.strip())

    assert(len(orig) == len(open(data_fp, "r").readlines()))
    assert(len(prep) == len(open(data_fp, "r").readlines()))

    return orig, prep


def assign_unk(tok):
    """
    Attribuer des tokens de mots inconnus
    """
    # chiffre
    if any(char.isdigit() for char in tok):
        return "--unk_digit--"

    # Ponctuation
    elif any(char in punct for char in tok):
        return "--unk_punct--"

    # Majuscule
    elif any(char.isupper() for char in tok):
        return "--unk_upper--"

    # Nouns
    elif any(tok.endswith(suffix) for suffix in noun_suffix):
        return "--unk_noun--"

    # Verbe
    elif any(tok.endswith(suffix) for suffix in verb_suffix):
        return "--unk_verb--"

    # Adjectif
    elif any(tok.endswith(suffix) for suffix in adj_suffix):
        return "--unk_adj--"

    # Adverbe
    elif any(tok.endswith(suffix) for suffix in adv_suffix):
        return "--unk_adv--"

    return "--unk--"


- A noter que Si _di_ est un dictionnaire, `key in di` renverra `True` si _di_ a une clé _key_, sinon `False`.

Le dictionnaire `vocab` utilisera ces fonctionnalités.

In [ ]:
# charge dans le corpus de train
with open("WSJ_02-21.pos", 'r') as f:
    training_corpus = f.readlines()

print(f"Quelques éléments de la liste des corpus de formation")
print(training_corpus[0:5])

Quelques éléments de la liste des corpus de formation
['In\tIN\n', 'an\tDT\n', 'Oct.\tNNP\n', '19\tCD\n', 'review\tNN\n']


In [ ]:
# lire les données de vocabulaire, séparées par chaque ligne de texte, et enregistrer la liste
with open("hmm_vocab.txt", 'r') as f:
    voc_l = f.read().split('\n')

print("Quelques éléments de la liste de vocabulaire")
print(voc_l[0:50])
print()
print("Quelques éléments à la fin de la liste de vocabulaire")
print(voc_l[-50:])

Quelques éléments de la liste de vocabulaire
['!', '#', '$', '%', '&', "'", "''", "'40s", "'60s", "'70s", "'80s", "'86", "'90s", "'N", "'S", "'d", "'em", "'ll", "'m", "'n'", "'re", "'s", "'til", "'ve", '(', ')', ',', '-', '--', '--n--', '--unk--', '--unk_adj--', '--unk_adv--', '--unk_digit--', '--unk_noun--', '--unk_punct--', '--unk_upper--', '--unk_verb--', '.', '...', '0.01', '0.0108', '0.02', '0.03', '0.05', '0.1', '0.10', '0.12', '0.13', '0.15']

Quelques éléments à la fin de la liste de vocabulaire
['yards', 'yardstick', 'year', 'year-ago', 'year-before', 'year-earlier', 'year-end', 'year-on-year', 'year-round', 'year-to-date', 'year-to-year', 'yearlong', 'yearly', 'years', 'yeast', 'yelled', 'yelling', 'yellow', 'yen', 'yes', 'yesterday', 'yet', 'yield', 'yielded', 'yielding', 'yields', 'you', 'young', 'younger', 'youngest', 'youngsters', 'your', 'yourself', 'youth', 'youthful', 'yuppie', 'yuppies', 'zero', 'zero-coupon', 'zeroing', 'zeros', 'zinc', 'zip', 'zombie', 'zone', 'zone

In [ ]:
# vocab : dictionnaire qui possède l'index des mots correspondants

vocab = {}

# Obtenez l'index des mots correspondants.
for i, word in enumerate(sorted(voc_l)):
    vocab[word] = i

print("Dictionnaire de vocabulaire, la clé est le mot, la valeur est un entier unique")
cnt = 0
for k,v in vocab.items():
    print(f"{k}:{v}")
    cnt += 1
    if cnt > 20:
        break

Dictionnaire de vocabulaire, la clé est le mot, la valeur est un entier unique
:0
!:1
#:2
$:3
%:4
&:5
':6
'':7
'40s:8
'60s:9
'70s:10
'80s:11
'86:12
'90s:13
'N:14
'S:15
'd:16
'em:17
'll:18
'm:19
'n':20


In [ ]:
# charge dans le corpus de test
with open("WSJ_24.pos", 'r') as f:
    y = f.readlines()

print("Un échantillon du corpus de tests")
print(y[0:10])

Un échantillon du corpus de tests
['The\tDT\n', 'economy\tNN\n', "'s\tPOS\n", 'temperature\tNN\n', 'will\tMD\n', 'be\tVB\n', 'taken\tVBN\n', 'from\tIN\n', 'several\tJJ\n', 'vantage\tNN\n']


In [ ]:
#corpus sans étiquettes, prétraité
_, prep = preprocess(vocab, "test.words")

print('La longueur du corpus de test prétraité : ', len(prep))
print('Ceci est un échantillon du test_corpus : ')
print(prep[0:10])

La longueur du corpus de test prétraité :  34199
Ceci est un échantillon du test_corpus : 
['The', 'economy', "'s", 'temperature', 'will', 'be', 'taken', 'from', 'several', '--unk--']


# <a name="2">II. Dictionnaires de Probabilité et de Tag (étiquette)</a>

A présent nous allons voir des dictionnaires nous permettant de calculer des probabilité que nous définirons plus précisément dans la prochaine partie avec les chaines de Markov

## <a name="tc">Transition_counts</a>

- Le premier dictionnaire est le dictionnaire `transition_counts` qui calcule le nombre de fois où chaque Pos Tag s'est produit à côté d'un autre Pos Tag.

Ce dictionnaire sera utilisé pour calculer:
$$ P (t_i | t_ {i-1})  $$

C'est la probabilité d'une étiquette à la position $ i $ étant donné l'étiquette à la position $ i-1 $.

Pour que nous puissions calculer l'équation 1, nous allons créer un dictionnaire `transition_counts` où
- Les clés sont `(prev_tag, tag)`
- Les valeurs sont le nombre de fois que ces deux tags sont apparues dans cet ordre.



## <a name="ec">Emission_counts</a>

Le deuxième dictionnaire que vous calculerez est le dictionnaire `émission_counts`. Ce dictionnaire sera utilisé pour calculer:

$$P(w_i | t_i)$$

En d'autres termes, nous l'utiliserons pour calculer la probabilité d'un mot compte tenu de son Tag.

Afin que nous puissions calculer l'équation 2, nous allons créer un dictionnaire `émission_counts` où
- Les clés sont `(tag, mot)`
- Les valeurs sont le nombre de fois où cette paire est apparue dans votre ensemble d'entraînement (formation).



## <a name="tac">Tag_counts</a>

Le dernier dictionnaire que nous calculerons est le dictionnaire `tag_counts`.
- La clé est l'étiquette le tag
- La valeur est le nombre de fois où chaque étiquette est apparue.



Ecrivons une fonction `create_dictionaries` qui prend dans le `training_corpus` et retourne les trois dictionnaires mentionnés ci-dessus`transition_counts`, `émission_counts` et` tag_counts`.
- `émission_counts`: associe (tag, word) au nombre de fois où cela s'est produit.
- `transition_counts`: aqqocie (prev_tag, tag) au nombre de fois où il est apparu.
- `tag_counts`: associe (tag) sur le nombre de fois où cela s'est produit.

Note d'implémentation: Cette routine utilise * defaultdict *, qui est une sous-classe de * dict *.
- Un dictionnaire Python standard lance un * KeyError * si vous essayez d'accéder à un élément avec une clé qui n'est pas actuellement dans le dictionnaire.
- En revanche, le * defaultdict * créera un élément du type de l'argument, dans ce cas un entier avec la valeur par défaut de 0.
- Voir [defaultdict](https://docs.python.org/3.3/library/collections.html#defaultdict-objects).

In [ ]:
def create_dictionaries(training_corpus, vocab) :
    """
    Entrée :
        training_corpus : un corpus où chaque ligne comporte un mot suivi de sa balise.
        vocab : un dictionnaire où les clés sont des mots du vocabulaire et la valeur est un index
    Sortie :
        emission_counts : un dictionnaire où les clés sont (tag, mot) et les valeurs sont les comptes
        transition_counts : un dictionnaire où les clés sont (prev_tag, tag) et les valeurs sont les comptes
        tag_counts : un dictionnaire où les clés sont les tags et les valeurs sont les comptes
    """

    # initialiser les dictionnaires en utilisant defaultdict
    emission_counts = defaultdict(int)
    transition_counts = defaultdict(int)
    tag_counts = defaultdict(int)

    # Initialisez "prev_tag" (balise précédente) avec l'état de départ, indiqué par '--s--'.
    prev_tag = "--s--"

    # utiliser le "i" pour suivre le numéro de ligne dans le corpus
    i = 0

    # Chaque élément du corpus de formation contient un mot et sa balise POS
    # Passez en revue chaque mot et sa balise dans le corpus de formation
    for word_tag in training_corpus:

        # Augmenter le nombre de word_tag
        i += 1

        # Tous les 50 000 mots, imprimez le nombre de mots
        if i % 50000 == 0 :
            print(f"nombre de mots = {i}")


        # obtenir le mot et le tag en utilisant la fonction d'aide get_word_tag
        word, tag = get_word_tag(word_tag,vocab)

        # Augmenter le nombre de transitions pour le mot et le tag précédents
        transition_counts[(prev_tag, tag)] += 1

        # Augmenter le nombre d'émissions pour le tag et le mot
        emission_counts[(tag, word)] += 1

        # Augmenter le nombre de tag
        tag_counts[tag] += 1

        # Mettre le tag précédent à ce tag (pour la prochaine itération de la boucle)
        prev_tag = tag



    return emission_counts, transition_counts, tag_counts

In [ ]:
emission_counts, transition_counts, tag_counts = create_dictionaries(training_corpus, vocab)

nombre de mots = 50000
nombre de mots = 100000
nombre de mots = 150000
nombre de mots = 200000
nombre de mots = 250000
nombre de mots = 300000
nombre de mots = 350000
nombre de mots = 400000
nombre de mots = 450000
nombre de mots = 500000
nombre de mots = 550000
nombre de mots = 600000
nombre de mots = 650000
nombre de mots = 700000
nombre de mots = 750000
nombre de mots = 800000
nombre de mots = 850000
nombre de mots = 900000
nombre de mots = 950000


In [ ]:
# obtenir tous les états de POS tag
states = sorted(tag_counts.keys())
print(f"Nombre de POS tags (nombre d'états') : {len(states)}")
print("Afficher ces POS tags (états)")
print(states)

Nombre de POS tags (nombre d'états') : 46
Afficher ces POS tags (états)
['#', '$', "''", '(', ')', ',', '--s--', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']


Les `états` sont les désignations des POS (Parts-of-speech) qui se trouvent dans les données de formation. Ils seront également appelés `tags` ou POS dans cette affectation.

- "NN" est un nom, au singulier,
- "NNS" est un nom, au pluriel.
- En outre, il existe des balises utiles comme "--s--" qui indiquent le début d'une phrase.
- Nous pouvons obtenir une description plus complète sur le site [Penn Treebank II tag set](https://www.clips.uantwerpen.be/pages/mbsp-tags).

In [ ]:
print("exemples de transition :  ")
for ex in list(transition_counts.items())[:3]:
    print(ex)
print()

print("exemples d'émissions :")
for ex in list(emission_counts.items())[200:203]:
    print (ex)
print()

print("exemple de mot ambigu : ")
for tup,cnt in emission_counts.items():
    if tup[1] == 'back': print (tup, cnt)

exemples de transition :  
(('--s--', 'IN'), 5050)
(('IN', 'DT'), 32364)
(('DT', 'NNP'), 9044)

exemples d'émissions :
(('DT', 'any'), 721)
(('NN', 'decrease'), 7)
(('NN', 'insider-trading'), 5)

exemple de mot ambigu : 
('RB', 'back') 304
('VB', 'back') 20
('RP', 'back') 84
('JJ', 'back') 25
('NN', 'back') 29
('VBP', 'back') 4


<a name='pred'></a>
## <a name="tp">Test et Précision</a>

Nous allons maintenant tester la précision des POS tag en utilisant notre dictionnaire `emission_counts`. Nous Implémentons `predict_pos` qui calcule la précision de notre modèle.
- Etant donné notre corpus de test prétraité `prep`, nous allons assigner un POS Tag à chaque mot de ce corpus.
- En utilisant le corpus de test original tag `y`, nous calculerons alors le pourcentage de tags que nous avons obtenu correctement.

- Pour attribuer un POS Tag à un mot, attribuons le POS tag le plus fréquent (plus grande valeur d'`emission_counts`) pour ce mot dans l'ensemble de formation (train).
- Ensuite, évaluons le fonctionnement de cette approche.  Chaque fois que nous faissons une prédiction basée sur le POS tag le plus fréquent pour le mot donné, vérifions si le POS tag réel de ce mot est le même.  Si c'est le cas, la prédiction est correcte !
- Calculons la précision en divisant le nombre de prédictions correctes par le nombre total de mots pour lesquels nous avons prédit le POS tag.

In [ ]:
def predict_pos(prep, y, emission_counts, vocab, states):
    '''
    Entrée :
        prep : une version prétraitée de "y". Une liste avec la composante "mot" des tuples.
        y : un corpus composé d'une liste de tuples où chaque tuple est constitué de (mot, POS)
        emission_counts : un dictionnaire dont les clés sont des tuples (tag,word) et la valeur est leurs nombres
        vocabulaire : un dictionnaire où les clés sont des mots de vocabulaire et la valeur est un index
        states : une liste triée de tous les tags possibles pour cette affectation
    Sortie :
        précision : Nombre de fois où vous avez classé un mot correctement
    '''

    # Initialiser le nombre de prédictions correctes à zéro
    num_correct = 0

    # Obtenir les tuples (tag, mot), stockés sous forme d'un ensemble
    all_words = set(emission_counts.keys())

    # Obtenir le nombre de tuples (mot, POS) dans le corpus 'y'.
    total = len(y)
    for word, y_tup in zip(prep, y):

        # Séparer la chaîne (mot, POS) en une liste de deux éléments
        y_tup_l = y_tup.split()

         # Vérifiez que y_tup contient à la fois le mot et le POS
        if len(y_tup_l) == 2:

            # Définir le véritable label POS pour ce mot (on le récupère)
            true_label = y_tup_l[1]

        else:
            # Si le y_tup ne contenait pas le mot et le POS, passez au mot suivant
            continue

        count_final = 0
        pos_final = ''

        # si le mot est dans le vocabulaire :
        if word in vocab:
            for pos in states:

                # définir la clé comme le tuple contenant le POS et le mot
                key = (pos,word)

                # vérifier si la clé (pos, word) existe dans le dictionnaire emission_counts
                if key in emission_counts: # complete this line

                 # obtenir le nombre d'émissions du tuple(pos,mot) pour chaque mot d'emissions count
                    count = emission_counts[key]

                   # garder la trace du POS avec le plus grand nombre
                    if count>count_final: # complete this line

                        # mettre à jour le décompte final (décompte le plus important)
                        count_final = count

                        # mettre à jour le POS final
                        pos_final = pos

            # Si le POS final (avec le plus grand nombre) correspond au vrai POS:
            if pos_final == true_label: # complete this line

                # Mettre à jour le nombre de prévisions correctes
                num_correct += 1


    accuracy = num_correct / total

    return accuracy

In [ ]:
accuracy_predict_pos = predict_pos(prep, y, emission_counts, vocab, states)
print(f"La précision de la prédiction à l'aide de predict_pos est {accuracy_predict_pos:.4f}")

La précision de la prédiction à l'aide de predict_pos est 0.8889


88,9 %, c'est vraiment bien. Avec les modèles markov cachés, nous devrions pouvoir obtenir une précision de **95%.**

# <a name="3">III. Les chaînes de Markov</a>



Exemple: Why not learn.......    

Devons nous compléter par un nom ? par un verbe, l'idée est que la proba que les parties du mots suivant soit l'étiquetage morpho-syntaxique dans une phrase, tend à dépendre du mot précédent, il s'agit d'un excellent exemple du fonctionnement des chaînes de Markov à très petite échelle. regardons cette représentation

<img src="images/cm1.png" style="width:700px;height:400;">

suivant le verbe **learn** on aura **20% de chance qu'il qoit suivi d'un verbe (verb)** et **60% de chance qu'il soit suivi d'un nom (noun)**.

### <a name="pré1">Présentation</a>

**Alors qu'est ce qu'une chaîne de Markov ?**

Il s'agit:

- D'un type de modèle stochastique qui décrit une séquence d'événements possibles
- Pour obtenir la probabilité de chaque événement, il n'a besoin que des états des événements précédents
- Le mot stochastique signifie simplement aléatoire ou aléatoire. Ainsi, un modèle stochastique incorpore et Les processus de modèles ont une composante aléatoire.


- Une chaîne de Markov peut être représentée sous la forme d'un graphe orienté
- Donc dans le contexte de l'informatique, un graphe est une sorte de structure de données qui est représenté visuellement comme un ensemble de cercles reliés par des lignes. Lorsque les lignes qui relient les cercles ont des flèches, cela indique une certaine direction, c'est ce qu'on appelle un graphe orienté. Les cercles du graphes représentent les états de notre modèle. Un état fait référence à une certaine condition du moment présent. Par exemple, si vous utilisez un graphique pour modéliser si l'eau est dans un état gelée, un état liquide ou un état gazeux, alors vous dessinez un cercle pour chacun de ces états pour représenter les trois états possibles dans lesquels l'eau peut être au moment présent on étiquette chaque état comme $q_1, q_2$ , $q_3$., etc. pour leur donner à chacun un nom unique, puis en se référant à l'ensemble de tous les états avec la lettre majuscule Q ou $Q=\{q_1,q_2,q_3\}$. Pour ce graphe suivant, il existe trois états, $q_1, q_2$ et $q_3$:.


<img src="images/cm2.png" style="width:700px;height:400;">

## <a name="cmems">Chaîne de Markov et étiquetage morpho-syntaxique</a>

### <a name="gpt">Graphe et Probabilité de transition</a>

Jusqu'à présent, vous avez vu les chaînes de Markov comme un graphe d'états et transitions entre ces états. Maintenant comment les utiliser pour certains étiquetage morpho-syntaxique:


Si vous considérez une phrase comme une séquence de mots avec des parties associées d'étiquetage morpho-syntaxique, vous pouvez représenter cette séquence avec un graphe où les étiquetages morpho-syntaxique sont des événements qui peuvent se représentés par les états de notre modèle de graphe .


<img src="images/cm3.png" style="width:700px;height:400;">

Dans cet exemple, NN est pour les noms, VB pour les verbes et O représentent toutes les autres étiquettes. Les arêtes (cercle bleus) du graphe ont des poids ou des probabilités de transition associés avec eux qui définissent la probabilité de passer d'un état à un autre.


Il y a une dernière propriété importante que possèdent les chaînes de Markov, appelée **propriété de Markov**, qui stipule que la probabilité de l'événement suivant ne dépend que des événements en cours. La propriété de Markov permet de garder le modèle simple en disant que tout ce dont vous avez besoin pour déterminer le prochain état est l'état actuel. Il n'est pas nécessaire d'obtenir des informations sur les états précédents.


Revenons à l'analogie de l'eau à l'état solide, liquide ou gazeux. Si nous regardons une tasse d'eau qui se trouve à l'extérieur, l'état actuel de l'eau est un état liquide. Lorsque vous modélisez la probabilité que l'eau dans la tasse passe à l'état gazeux, vous n'avez pas besoin de connaître l'historique de l'eau. Qu'elle provienne de glaçons ou de nuages de pluie. C'est logique, n'est-ce pas ?

Revenons sur l'exemple de phrase: **Why not learn.......**  . Si nous regardons à nouveau cette phrase et que nous voulons connaître la probabilité que le prochain mot qui suit learn soit un nom, alors cela dépend simplement de l'état dans lequel nous nous trouvons. Dans ce cas, les états du verbe sont désignés par VB parce que le mot courant appris est un verbe, donc la probabilité que le prochain mot soit un nom est la probabilité de transition pour passer du verbe au nom et aux états finaux.


La probabilité de transition est écrite sur la flèche qui va de VB à NN. Et comme nous pouvons le voir, elle est de 0,4.

### <a name="mt">Matrice de Transition </a>

Nous pouvons également utiliser un tableau pour stocker les états et les probabilités de transition. Un tableau est une représentation équivalente, mais plus compacte, du modèle de la chaîne de Markov. Et ce tableau est appelé une **matrice de transition** $A=(a_{ij})_{1 \leq i,j \leq N}$. Une matrice de transition est une matrice N par N, N étant le nombre d'états dans le graphe. Chaque ligne de la matrice représente les probabilités de transition d'un état vers tous les autres états. La matrice de transition peut etre de taille N+1 par N si notre premiere ligne correspond au probabilité initiale( état 0).

$$A = \left[
\begin{array}{cccc}
 a_{1,1}& \ldots  & a_{1,N}  \\
\vdots & \ddots & \vdots \\
a_{N+1,1} & \ldots  & a_{N+1,N}
\end{array}
\right]
$$

Par exemple dans la figure suivante, la première ligne représente le cas où l'état actuel est un nom. Les colonnes représentent les états futurs possibles qui pourraient venir ensuite. Les valeurs à l'intérieur du tableau représentent la probabilité de transition d'un nom à un autre d'un nom à un verbe et d'un nom à d'autres états. Notez que pour toutes les probabilités de transition d'un état donné, la somme de ces probabilités de transition doit toujours être égale à 1: $\sum_{i=1}^{N} a_{ij}=1,  \forall j \leq N$, de même, dans la matrice de transition, toutes les probabilités de transition de chaque ligne doivent s'additionner pour donner 1: $\sum_{j=1}^{N}a_{ij}=1,  \forall i \leq N$.

<img src="images/cm4.png" style="width:700px;height:400;">

## <a name="mmc">Modèle de Markov cachés</a>

### <a name="pré2">Présentation</a>

Pour décoder les états cachés d'un mot nous pouvons utiliser

les modèles de Markov cachés et vous les utiliserez pour décoder les états cachés d'un mot. Dans notre cas, les états cachés ne sont que l'étiquetage morpho-syntaxique (POS tag) de ce mot.

Le nom de modèle de Markov caché implique que les états sont cachés ou ne sont pas directement observables. Pour revenir au modèle de Markov qui comporte les états pour l'étiquetage morpho-syntaxiques, comme le nom NN, le verbe VB ou autre O, nous pouvons maintenant les considérer comme des **états cachés (Hidden States)** parce qu'ils ne sont pas directement observables à partir des données textuelles.

Il peut sembler un peu déroutant de penser que ces données sont cachées parce que si vous regardez un certain mot comme "jump", en tant qu'humain familier de la langue anglaise, vous pouvez voir qu'il s'agit d'un verbe.

Du point de vue d'une machine, cependant, elle ne voit que le jump sous forme de texte et elle ne sait pas s'il s'agit d'un verbe ou d'un nom. Pour une machine qui regarde les données du texte, ce qu'elle va observer, ce sont les mots réels, comme jump, run et fly. On dit que ces mots sont **observables** parce qu'ils peuvent être vus par la machine.


Le modèle de la chaîne de Markov et le modèle de Markov caché ont des probabilités de transition, qui peuvent être représentées par une matrice A de dimensions N+1 par N où N est le nombre d'états cachés.


### <a name="pme">Probabilité et matrice d'émission</a>

Le modèle de Markov caché a également des probabilités supplémentaires connues sous le nom de **probabilités d'émission**. Celles-ci décrivent la transition entre les états cachés de votre modèle de Markov caché, qui concernent l'étiquetage morpho-syntaxique vues ici comme des cercles pour les noms, verbes et autres, et les observables ou les mots de votre corpus, représentés ici à l'intérieur de rectangles.

<img src="images/cm5.png" style="width:700px;height:400;">

Voici ci dessus les observables pour les états cachés VB, qui sont les mots going, to, eat. La probabilité d'émission du verbe aux états cachés vers l'observable eat est de 0,5. Cela signifie que lorsque le modèle est actuellement à l'état caché pour un verbe, il y a 50% de chances que l'observable que le modèle va émettre soit le mot eat.






Voici une représentation équivalente des probabilités d'émission sous la forme d'un tableau:

<img src="images/cm6.png" style="width:700px;height:400;">

Chaque ligne est désignée pour l'un des états cachés. Une colonne est désignée pour chacun des observables. Par exemple, la ligne pour le verbe d'état caché croise la colonne pour l'observable eat. La valeur, 0,5, est la probabilité d'émission de passer du verbe d'état à l'émission du eat observable. La **matrice d'émission B** représente les probabilités de transition de vos n états cachés représentant nos étiquetages morpho-syntaxique vers les n mots de notre corpus. Là encore, la somme des probabilités de la ligne est de 1 et ce que nous avons pu constater dans cet exemple est qu'il existe des probabilités d'émission supérieures à 0 pour nos trois étiquetages morpho-syntaxique. Cela s'explique par le fait que les mots peuvent avoir des étiquetages morpho-syntaxique et des signes différents selon le contexte dans lequel ils apparaissent.

Par exemple, He lay on his **back** et I'll be **back**, le mot **back** doit avoir un étiquetage morpho-syntaxique dans chacune des phrases. Le nom  pour la phrase, He lay on his **back** (il s'allonge sur le dos) , et l'adverbe pour, I'll be **back**, (je reviendrai). Un rapide rappel des modèles de Markov cachés. Ils consistent en un ensemble de N états, Q. La matrice de transition A a la dimension N par N et la matrice d'émission B a la dimension N par |V| taille du vocabulaire.

### <a name="ecpt">Exemple calcul de probabilité de translation </a>

Soit le corpus suivant composé des 3 phrases

- You eat
- The Gatmeal
- You eat

<img src="images/cm7.png">

En considérant que les rectangles de couleurs correpondent à des étiquettes la probabilité de Transition de **You** dans l'étiquette bleue (la paire (bleu, You)) est 2/3 car le bleu entoure 3 mot du corpus et la paire (bleu, You) apparait 2 fois comme on peut le voir ci dessus.

La procédure est la suivante :
- Compter le nombre d'occurence des pairs d'étiquettes $C(t_{i-1},t_i)$
- Calculer la proba $$P(t_i \mid t_{i-1}) = \frac{C(t_{i-1},t_i)}{\sum_{j=1}^{N}C(t_{i-1},t_j)}$$

Exemple on considère le corpus suivant

<s> in a station of the metro   (Dans une station de métro)
<s> the apparition of thes faces in the crowd  (l'apparition de ces visages dans la foule)
<s> petals on a wet black (des pétales sur un noir humide)

on considere que le caractère < s > appartient à l'étiquette $\pi$ l'image suivante nous montre les étapes pour construire la matrice des nombres d'occurences $C(t_{i-1},t_i)$ nous remarquons qu'il n'y a **aucun verbe et la ponctuation compte**.

Voici en image les étapes qui consistent à remplir chaque coefficient de la matrice chaque mot est une couleur associé à son POS tag,  le nombre de paire de mot encadré en gris correspond au nombre encadré en noire de la matrice A:

<img src="images/cm8.jpg" style="width:700px;height:400;">

Désormais on ajoute une colonne pour calculer $\sum_{j=1}^{N}C(t_{i-1},t_j)=C(t_{i-1})$ le nombre de paire commencant par le tag $t_{i-1}$ en additionnant sur chaque ligne et on pourra ensuite calculer $P(t_i \mid t_{i-1})$

<img src="images/cm9.png" style="width:700px;height:400;">

On remarque beaucoup de 0 dans notre matrice à cause de l'absence de verbe et celà pourrait poser problème dans les calculs, on utilisera donc un **smoothing** de tel sorte qu'on ait: $$P(t_i \mid t_{i-1}) = \frac{C(t_{i-1},t_i)+ \epsilon}{\sum_{j=1}^{N}C(t_{i-1},t_j)+N*\epsilon}$$

avec N=3 la taille du corpus car 3 étiquettes sans $\pi$
et

<img src="images/cm10.png" style="width:700px;height:400;">

On obtient notre matice de transition final en divisant chaque élément par l'élément de la meme ligne placé sur la dernière colonne que nous avons ajouté

<img src="images/cm11.png">

## <a name="rp">Retour au Problème</a>

### <a name="cmta">Création de la matrice de transition A</a>

Retour sur nos données maintenant que nous avons nos `emission_counts`, `transition_counts`, et `tag_counts`, nous allons commencer à implémenter le Modèle de Markov Caché.

Cela nous permettra de construire rapidement
- Une matrice de probabilité de transition "A".
- Une matrice de probabilité d'émission "B".

Nous utiliserons également un certain lissage(smoothing) lors du calcul de ces matrices
On commence d'abord par la **matrice de transition A**.

On rappelle que :

$$P(t_i | t_{i-1}) = \frac{C(t_{i-1}, t_{i}) + \epsilon }{C(t_{i-1}) +N*\epsilon  }$$

- $N$ est le nombre total d'étiquettes
- $C(t_{i-1}, t_{i})$ est le comptage du n-uplet (POS précédent, POS actuel) dans le dictionnaire `transition_counts`.
- $C(t_{i-1})$ est le nombre de POS tag précédent dans le dictionnaire `tag_counts`.
- $\epsilon$ est le paramètre de lissage (smoothing) vu dans l'[exemple](#empt).

<a name='trans'></a>
Implémentons la fonction `create_transition_matrix` ci-dessous pour tous les tags. Votre tâche est de produire une matrice qui calcule l'équation 3 pour chaque cellule de la matrice `A`.

In [ ]:
def create_transition_matrix(alpha, tag_counts, transition_counts):
    '''
    Entrée :
        alpha : numéro utilisé pour le lissage
        tag_counts : un dictionnaire qui associe chaque balise à son compte respectif
        transition_counts : compte de transition pour le mot et la balise précédents
    Sortie :
        A : matrice de dimension (num_tags,num_tags)
    '''

    # Obtenir une liste triée de POS tags uniques
    all_tags = sorted(tag_counts.keys())

    # Comptez le nombre de POS tag  uniques
    num_tags = len(all_tags)

    # Initialiser la matrice de transition "A".
    A = np.zeros((num_tags,num_tags))

    # Obtenir les tuples de transition uniques (POS précédent, POS actuel)
    trans_keys = set(transition_counts.keys())

    # Parcourir chaque ligne de la matrice de transition A
    for i in range(num_tags):

        # Passez en revue chaque ligne de la matrice de transition A
        for j in range(num_tags):

            # Initialiser le comptagee de (POS précédent, POS actuel) à zéro
            count = 0

            # Définir le n-uplet (POS précédent, POS actuel)
            # Obtenir le tag à la position i et le tag à la position j (de la liste all_tags)
            key = (all_tags[i],all_tags[j])

            # Vérifiez si le tuple (POS précédent, POS actuel)
            # existe dans le dictionnaire des comptes de transition
            if transition_counts:

                # Obtenez le compte du dictionnaire transition_counts
                # pour le tuple (POS précédent, POS actuel)
                count = transition_counts[key]

            # Obtenir le nombre de tags précédents (position i de l'index) à partir de tag_counts
            count_prev_tag = tag_counts[all_tags[i]]


            # Appliquer le lissage en utilisant le compte du tuple, alpha,
            # nombre du précédent tag, alpha, et nombre total de balises
            A[i,j] = (count + alpha) / (count_prev_tag + alpha*num_tags)

    return A

In [ ]:
alpha = 0.001
A = create_transition_matrix(alpha, tag_counts, transition_counts)
# Tester votre fonction
print(f"A à la ligne 0, col 0: {A[0,0]:.9f}")
print(f"A at ligne 3, col 1: {A[3,1]:.4f}")

print("Sous partie de la matrice de transition A")
A_sub = pd.DataFrame(A[30:35,30:35], index=states[30:35], columns = states[30:35] )
print(A_sub)

A à la ligne 0, col 0: 0.000007040
A at ligne 3, col 1: 0.1691
Sous partie de la matrice de transition A
              RBS            RP           SYM        TO            UH
RBS  2.217069e-06  2.217069e-06  2.217069e-06  0.008870  2.217069e-06
RP   3.756509e-07  7.516775e-04  3.756509e-07  0.051089  3.756509e-07
SYM  1.722772e-05  1.722772e-05  1.722772e-05  0.000017  1.722772e-05
TO   4.477336e-05  4.472863e-08  4.472863e-08  0.000090  4.477336e-05
UH   1.030439e-05  1.030439e-05  1.030439e-05  0.061837  3.092348e-02


### <a name="cmeb">Création de la matrice d'émission B</a>


$$P(w_i | t_i) = \frac{C(t_i, mot_i)+ \epsilon}{C(t_{i}) +N*\epsilon }$$

- $C(t_i, mot_i)$ est le nombre de fois que $mot_i$ a été associé au $tag_i$ dans les données de train (stockées dans le dictionnaire `emission_counts`).
- $C(t_i)$ est le nombre de fois où $tag_i$ figurait dans les données  train (stockées dans le dictionnaire `tag_counts`).
- $N$ est le nombre de mots dans le vocabulaire
- $\epsilon$ est un paramètre de lissage (smoothing).

Dans notre exemple de corpus à 3 phrases prenons le mot in nous avons la figure suivante :
<img src="images/cm12.jpg" style="width:700px;height:400;">

Dans la figure ci dessus on voit que le mot **in** apparait deux fois comme un autre **Tag**.

<a name='emiss'></a>
Revenons à nos données on va implémenter une fonction `create_emission_matrix` ci-dessous qui calcule la matrice des probabilités d'émission `B`. Notre fonction prend en compte $\alpha$, le paramètre de lissage, `tag_counts`, qui est un dictionnaire mettant en correspondance chaque étiquette avec son compte respectif, le dictionnaire `emission_counts` où les clés sont (étiquette, mot) et les valeurs sont les comptes. Notre tâche consiste à produire une matrice qui calcule $P(w_i | t_i)$ pour chaque cellule de la matrice "B".

In [ ]:
def create_emission_matrix(alpha, tag_counts, emission_counts, vocab):
    '''
    Entrée :
        alpha : paramètre de réglage utilisé dans le lissage
        tag_counts : un dictionnaire qui associe chaque balise à son compte respectif
        emission_counts : un dictionnaire où les clés sont (tag, mot) et les valeurs sont les comptes (leur nombres)
        vocab : un dictionnaire où les clés sont des mots de vocabulaire et la valeur est un index
    Sortie :
        B : une matrice de dimension (num_tags, len(vocab))
    '''

    # obtenir le nombre POS tag
    num_tags = len(tag_counts)

    # Obtenir une liste de tous les POS tags
    all_tags = sorted(tag_counts.keys())

    # Obtenir le nombre total de mots uniques dans le vocabulaire
    num_words = len(vocab)

    # Initialiser la matrice d'émission B avec des places pour
    # les tags dans les lignes et les mots dans les colonnes
    B = np.zeros((num_tags, num_words))

    # Obtenir un ensemble de toutes les tuples (POS, mot)
    # à partir des clés du dictionnaire emission_counts
    emis_keys = set(list(emission_counts.keys()))

    # Passez par chaque ligne (POS tags)
    for i in range(num_tags): # complete this line

        # Parcourir chaque colonne (mots)
        for j in range(num_words): # complete this line

             # Initialiser le comptagee de (POS précédent, POS actuel) à zéro
            count = 0

            # Define the (POS tag, word) tuple for this row and column
            # Définissez le tuple (POS tag, word) pour cette ligne et cette colonne
            key =  (all_tags[i],vocab[j])

            # vérifier si le n-uplet (POS tag, word) existe comme clé dans les comptes d'émissions
            if key in emission_counts.keys(): # complete this line

                # Obtenir le nombre de (POS tag, word) à partir de emission_counts d
                count = emission_counts[key]

            # Obtenir le nombre de POS tag
            count_tag = tag_counts[all_tags[i]]

            # Appliquer le lissage et stocker la valeur lissée (smoothed)
            # dans la matrice d'émission B pour cette ligne et cette colonne
            B[i,j] = (count + alpha) / (count_tag+ alpha*num_words)


    return B

In [ ]:
# créons notre matrice de probabilité d'émission. Cela prend quelques minutes.
B = create_emission_matrix(alpha, tag_counts, emission_counts, list(vocab))

print(f"Voir la position de la matrice à la ligne 0, colonne 0 : {B[0,0] :.9f}")
print(f"Voir la position de la matrice à la ligne 3, colonne 1 : {B[3,1] :.9f}")

# Essayons de visualiser les émissions pour quelques mots dans un exemple de cadre de données
cidx  = ['725','adroitly','engineers', 'promoted', 'synergy']

# Obtenons l'identifiant de l'entier pour chaque mot
cols = [vocab[a] for a in cidx]

# Choisissons les POS tag à afficher dans un exemple de cadre de données
rvals =['CD','NN','NNS', 'VB','RB','RP']

# Pour chaque POS tag, obtenez le numéro de ligne de la liste des "états".
rows = [states.index(a) for a in rvals]

# Obtenir les émissions pour l'échantillon de mots, et l'échantillon de POS tag
B_sub = pd.DataFrame(B[np.ix_(rows,cols)], index=rvals, columns = cidx )
print(B_sub)

Voir la position de la matrice à la ligne 0, colonne 0 : 0.000006032
Voir la position de la matrice à la ligne 3, colonne 1 : 0.000000720
              725      adroitly     engineers      promoted       synergy
CD   8.201296e-05  2.732854e-08  2.732854e-08  2.732854e-08  2.732854e-08
NN   7.521128e-09  7.521128e-09  7.521128e-09  7.521128e-09  2.257091e-05
NNS  1.670013e-08  1.670013e-08  4.676203e-04  1.670013e-08  1.670013e-08
VB   3.779036e-08  3.779036e-08  3.779036e-08  3.779036e-08  3.779036e-08
RB   3.226454e-08  6.456135e-05  3.226454e-08  3.226454e-08  3.226454e-08
RP   3.723317e-07  3.723317e-07  3.723317e-07  3.723317e-07  3.723317e-07


# <a name="4">IV. Algorithme de Viterbi </a>

Jusqu'à présent, nous avons calculé les probabilités de transition et d'émission pour la chaîne de Markov et le modèle de Markov caché. Étant donné les POS tag dans ces probabilités, nous pouvons facilement sélectionner le POS tag suivant le plus probable ou le mot le plus probable. Pour ce faire, il suffit de rechercher l'entrée correcte dans la ligne correspondante de la matrice de transition ou d'émission.

Mais si l'on nous donne une phrase, **Why not learn something ?** Quelle est la séquence la plus probable de POS tag étant donné la phrase dans notre modèle ? La séquence peut être calculée en utilisant l'**algorithme de Viterbi**.



Nous allons voir beaucoup de formules qui sont toutes basées sur , mais l'algorithme de Viterbi est un algorithme de graphe basée sur des matrices représentant notre modèle de Markov caché.


En imaginant le problème que nous voulons résoudre sur le graphe, il nous sera beaucoup plus facile de comprendre les formules et l'algorithme. Examinons ce modèle-jouet et la phrase **"< s > I love to learn"**. Avec un token de départ, vous voulez trouver la séquence d'états cachés ou de Pos tag qui ont la probabilité la plus élevée pour cette séquence.


Sachons que le mot **love** peut être émis à la fois par les états nominaux, NN et les états verbaux, NN.


En regardant **la figure ci dessous** en numérotant nos graohe de 1 à 6 du plus haut dans la figure au plus bas

- Graphe 1: Nous partons des états initiaux en sélectionnant les états cachés les plus probables suivants, ici l'état O, car le mot I ne peut être émis par aucun autre état dans ce modèle-jouet. Cela implique les probabilités de transition indiquées en vert avec 0,3 et la probabilité d'émission en orange avec 0,5. La probabilité conjointe d'observation du mot I et d'une transition par l'état O est de 0,15, que nous pouvons calculer en multipliant la probabilité de transition, 0,3 et la probabilité d'émission de 0,5.
<img src="images/vit1.png" style="width:700px;height:400;">
- Graphe 2: Maintenant, il y a deux possibilités d'avoir observé le mot **love**. C'est soit en passant par les états cachés, NN ou les états cachés, VB. Les probabilités de transition sont les mêmes pour aller vers l'un ou l'autre des deux états cachés.
<img src="images/vit2a.png" style="width:700px;height:400;">
- Graphe 3: La probabilité d'émission du mot love est plus élevée à partir des états VB, nous devons donc choisir ce chemin avec une probabilité combinée de 0,25=0.5*0.5.
<img src="images/vit2.png" style="width:700px;height:400;">
- Graphe 4: Ensuite, nous revenons à l'état O car il n'y a pas d'autres états cachés avec une probabilité d'émission non nulle pour le mot **to**. La probabilité combinée est ici de 0,08.
<img src="images/vit3.png" style="width:700px;height:400;">
- Graphe 5: Enfin, nous revenons aux états VB car il n'y a pas d'autre option pour ce modèle de jeu. Cette étape a une probabilité combinée de 0,1.
<img src="images/vit4.png" style="width:700px;height:400;">
- Graphe 6: La probabilité totale est le produit de toutes les probabilités pour les étapes individuelles que nous avons choisies, qui est de 0,0003 ici.
<img src="images/vit5.png" style="width:700px;height:400;">



L'algorithme de Viterbi calcule en fait plusieurs de ces chemins en même temps afin de trouver la séquence d'états cachés la plus probable. Il utilise la représentation matricielle du modèle de Markov caché. L'algorithme peut être divisé en trois étapes principales : l'étape d'initialisation, la passe en avant (Forward) et la passe en arrière (Backward). Compte tenu de nos probabilités de transition et d'émission, nous commençons par remplir puis utiliser les matrices auxiliaires C et D. La matrice C contient les probabilités optimales intermédiaires et la matrice D, les indices des états visités. En parcourant le graphique du modèle pour trouver la séquence la plus probable de POS tag pour la séquence de mots donnée, $w_1$ jusqu'à $w_K$. Ces deux matrices ont n lignes, où n est le nombre de POS Tag ou d'états cachés dans notre modèle, et k colonnes, où k est le nombre de mots dans la séquence donnée.

<img src="images/viterbi2.png" >

## <a name="ini">Initialisation </a>

Comment pouvons nous  initialiser une matrice qui peut nous indiquer les POS tag de chaque mot. Cette matrice nous indiquera la probabilité que chaque mot appartienne à une certaine POS Tag.

Voyons comment nous pouvons faire. Comme vous venez de le voir, l'étape d'initialisation est l'une des trois étapes permettant de remplir les matrices auxiliaires, C et D.

- Dans l'étape d'initialisation, la première colonne de chacune de nos matrices, C et D, est remplie. La première colonne de C représente la probabilité des transitions des états de départ dans le graphe vers le premier tag $t_i$ et le mot $w_1$, ce qui signifie que nous essayons de passer du tag 1 au mot $w_1$. Par souci de clarté, nous présentons ici un modèle avec trois états cachés. Les entrées de la première colonne, $c_{i,1}$, sont donc les produits des probabilités de transition des états initiaux et des probabilités d'émission respectives. Comme nos probabilités initiales sont contenues dans la première ligne de la matrice de transition A, c'est la même chose que $a_{1,i}$ fois la probabilité d'émission correspondante b. La fonction **cindex** renvoie simplement l'index de colonne dans C pour le mot donné ici, $w_1$ l'indice est 1 .
<img src="images/init1.png" style="width:700px;height:400;">
- Dans la matrice D, nous stockons les étiquettes qui représentent les différents états que nous traversons lorsque nous trouvons la séquence la plus probable de Pos tag pour la séquence de mots donnée, $w_1$ jusqu'à $w_K$. Dans la première colonne, il suffit de mettre toutes les entrées à zéro, car il n'y a pas de parties de POS Tag que nous avons traversées. C'est tout pour la première étape. Nous savons maintenant comment initialiser nos matrices.
<img src="images/init2.png" style="width:700px;height:400;">


Dans cette partie nous implémenterons l'algorithme de Viterbi qui utilise la programmation dynamique. Plus précisément, nous utiliserons vos deux matrices, **A** de [`create_transition_matrix`](#trans) et **B** de [`create_emission_matrix`](#emiss)  , pour calculer l'algorithme de Viterbi. Nous avons décomposé ce processus en trois étapes principales pour vous.

* **Initialisation** - Dans cette partie, vous initialisez les matrices `best_path` et `best_probs` que vous allez remplir dans `feed_forward`.
* **forward** - A chaque étape, vous calculez la probabilité que chaque chemin se produise et les meilleurs chemins jusqu'à ce point.
* **backward** : Cela vous permet de trouver le meilleur chemin avec les plus grandes probabilités.



Nous commencerons par initialiser deux matrices de même dimension.

- best_probs : Chaque cellule contient la probabilité de passer d'une balise POS à un mot du corpus. (Matrice **C**).

- best_paths : Une matrice qui vous aide à tracer le meilleur chemin possible dans le corpus.  (Matrice **D**).


Ecrivons une fonction `initialize` ci-dessous qui initialise les matrices `best_probs` et `best_path`.

Les deux matrices seront initialisées à zéro, sauf la colonne zéro de `best_probs`.  
- La colonne zéro de `best_probs` est initialisée avec l'hypothèse que le premier mot du corpus a été précédé par un token de départ ("--s--").
- Cela vous permet de faire référence à la matrice **A** pour la probabilité de transition

Voici comment initialiser la colonne 0 de `best_probs` :
- La probabilité que le meilleur chemin allant de l'index de départ à un POS tag donnée indexée par un entier $i$ est dénotée par $\textrm{best_probs}[s_{idx}, i]$.
- Ceci est estimé comme la probabilité que le tag de départ passe au POS tag indexée par l'indice $i$ : $\mathbf{A}[s_{idx}, i]$ ET que le POS Tag indexée par $i$ émette le premier mot du corpus donné , qui est $\mathbf{B}[i, vocab[corpus[0]]]$.
- Notez que vocab[corpus[0]] fait référence au premier mot du corpus (le mot en position 0 du corpus donc $w_0$).
- **vocab** est un dictionnaire qui renvoie l'entier unique qui se réfère à ce mot particulier.

Conceptuellement, il ressemble à ceci :
$\textrm{best_probs}[s_{idx}, i] = \mathbf{A}[s_{idx}, i] \times \mathbf{B}[i, corpus[0] ]$


Pour éviter de multiplier et de stocker de petites valeurs sur l'ordinateur, nous prendrons le log du produit, qui devient la somme de deux logs :

$best\_probs[i,0] = log(A[s_{idx}, i]) + log(B[i, vocab[corpus[0]]$

De plus, pour éviter de prendre le log de 0 (qui est défini comme l'infini négatif), le code lui-même se contentera de mettre $best\_probs[i,0] = float('-inf')$ quand $A[s_{idx}, i] == 0$


L'implémentation pour initialiser $best\_probs$ ressemble donc à ceci :

$ si A[s_{idx}, i] <> 0 : best\_probs[i,0] = log(A[s_{idx}, i]) + log(B[i, vocab[corpus[0]]))$

$ si A[s_{idx}, i] == 0 : best\_probs[i,0] = float('-inf')$

In [ ]:
def initialize(states, tag_counts, A, B, corpus, vocab):
    '''
    Entrée :
        states : une liste de toutes les parties possibles du POS
        tag_counts : un dictionnaire qui associe chaque balise à son compte respectif
        A : Matrice de transition de dimension (num_tags, num_tags) num_tags : nombre de tags
        B : Matrice d'émission de dimension (num_tags, len(vocab))
        corpus : une séquence de mots dont le POS doit être identifié dans une liste
        vocabulaire : un dictionnaire où les clés sont des mots de vocabulaire et la valeur est un index
    Sortie :
        best_probs : matrice de la dimension (num_tags, len(corpus)) des flottants
        best_paths : matrice de la dimension (num_tags, len(corpus)) des entiers
    '''
    # Obtenir le nombre total de balises uniques sur le point de vente
    num_tags = len(tag_counts)

    # Initialiser la matrice best_probs
    # POS tag dans les lignes, nombre de mots dans le corpus comme les colonnes
    best_probs = np.zeros((num_tags, len(corpus)))

    # Initialiser la matrice des meilleurs chemins
    # POS tag dans les lignes, nombre de mots dans le corpus en colonnes
    best_paths = np.zeros((num_tags, len(corpus)), dtype=int)

    # Définir le token de départ
    s_idx = states.index("--s--")


    # Passez en revue chacune des POS
    for i in range(num_tags) :

        # Gérer le cas particulier où la transition du token de départ au POS tag i est nulle
        if A[s_idx,i] == 0 : # complétez cette ligne

            # Initialiser les best_probs sur le tag POS 'i', colonne 0, à l'infini négatif ou moins l'infini
            best_probs[i,0] = float('-inf')

        # Pour tous les autres cas où la transition du token de départ au POS Tag i est non nulle :
        else :

            # Initialiser best_probs au niveau du tag POS "i", colonne 0
            # Vérifiez la formule dans les instructions ci-dessus
            best_probs[i,0] = math.log(A[s_idx,i]) + math.log(B[i,vocab[corpus[0]]])

    return best_probs, best_paths

In [ ]:
best_probs, best_paths = initialize(states, tag_counts, A, B, prep, vocab)

In [ ]:
# Test the function
print(f"best_probs[0,0]: {best_probs[0,0]:.4f}")
print(f"best_paths[2,3]: {best_paths[2,3]:.4f}")

best_probs[0,0]: -22.6098
best_paths[2,3]: 0.0000


## <a name="vf">Viterbi forward</a>

- Nous allons voir comment nous pouvons attribuer une étiquette (POS tag) à certains mots de la phrase. Voyons un exemple. **La passe en avant (forward)** est la deuxième des trois étapes pour remplir nos matrices C et D. Maintenant que nous avons initialisé les matrices C et D, toutes les entrées restantes dans les deux matrices, C et D, sont remplies colonne par colonne pendant la passe en avant. Pour la matrice C, les entrées sont calculées par cette fonction d'apparence compliquée: $$c_{i,j}= \max_{{k}} c_{k,j-1}*a_{k,i} b_{i,cindex(w_j)}$$.
<img src="images/fp1.png" style="width:700px;height:400;">


- Imaginons l'étape suivante sur un autre graphe, pour que cela devienne clair. Supposons que nous voulions calculer l'entrée $c_{1,2}$. Ensuite, nous pouvons compléter les valeurs de la formule à partir du dernier terme $b_{1,cindex(w_2)}$. Le dernier terme de la formule est simplement la probabilité d'émission de l'étiquette $t_1$ vers $w_2$. Nous avons le $a_{k,1}$, qui est la probabilité de transition de l'étiquette $t_k$ vers le le Pos tag actuel $t_1$ et $c_{k,1}$, qui représente la probabilité pour le chemin précédent que nous avons parcouru. Nous choisissons le k, qui maximise la formule entière. Dans ce cas, il y a trois états qui ne sont pas l'état initial. Ainsi, k est soit 1, 2 ou 3.

<img src="images/fp2.png" style="width:700px;height:400;">

- Dans chaque $d_{i,j}$, nous sauvegardons simplement le k, celui qui maximise l'entrée dans c_{i,j}. Dans ce cas, il y a trois états qui ne sont pas l'état initial, donc k est soit 1, 2 ou 3. Ceci est défini par cette formule effrayante: $$d_{i,j}= \arg\max_{{k}} c_{k,j-1}*a_{k,i} b_{i,cindex(w_j)}$$, qui est la même que pour $c_{i,j}$ , à l'exception de l'argmax de tête. La fonction argmax renvoie le k, qui maximise les arguments de la fonction au lieu de la valeur maximale. Il ne reste presque plus qu'un pas à franchir. Nous calculons maintenant la matrice de probabilité à l'aide de l'algorithme de Viterbi. Prochainement, nous allons voir comment nous pouvons utiliser cette matrice de probabilité que nous venons de créer pour reconstruire le chemin (backward), afin de pouvoir identifier chaque mot avec les POS tag.




<img src="images/fp3.png" style="width:700px;height:400;">



Revenons à nos donnée, nous implémentons la fonction `viterbi_forward`. En d'autres termes, nous remplirons nos matrices `best_probs` et `best_paths` qui correspondent à C et D.
- Avancez à travers le corpus.
- Pour chaque mot, calculez une probabilité pour chaque tag possible.
- Contrairement à l'algorithme précédent, [`predict_pos`](#pred), celui ci inclura le chemin jusqu'à cette combinaison (mot, tag).

La formule pour calculer la probabilité et le chemin pour le $i^{ème}$ mot dans le corpus, le mot précédent $i-1$ dans le corpus, le POS tag actuel $j$, et le POS tag précédent $k$ est la suivante

$\mathrm{prob} = \mathbf{best\_prob}_{k, i-1} + \mathrm{log}(\mathbf{A}_{k, j}) + \mathrm{log}(\mathbf{B}_{j, vocab(corpus_{i})})$.

où $corpus_{i}$ est le mot dans le corpus à l'index $i$, et $vocab$ est le dictionnaire qui obtient l'entier unique qui représente un mot donné.

$\mathrm{path} = k$

où $k$ est l'entier représentant le POS tag précédent.


In [ ]:
def viterbi_forward(A, B, test_corpus, best_probs, best_paths, vocab) :
    '''
    Entrée :
        A, B : Les matrices de transiton et d'émission respectivement
        test_corpus : une liste contenant un corpus prétraité
        best_probs : une matrice initialisée de dimension (num_tags, len(corpus))
        best_paths : une matrice initialisée de dimension (num_tags, len(corpus))
        vocabulaire : un dictionnaire où les clés sont des mots de vocabulaire et la valeur est un index
    Sortie :
        best_probs : une matrice complète de dimension (num_tags, len(corpus))
        best_paths : une matrice complète de dimension (num_tags, len(corpus))
    '''
    # Obtenir le nombre de POS tag uniques (qui est le nombre de lignes dans best_probs)
    num_tags = best_probs.shape[0]

    # Passez en revue chaque mot du corpus à partir du mot 1
    # Rappelez-vous que le mot 0 a été initialisé dans `initialize()`
    for i in range(1, len(test_corpus)):

        # Nombre de mots traités, tous les 5000 mots
        if i % 5000 == 0 :
            print("Mots traités : {:>8}".format(i))


        # Pour chaque POS tag unique que le mot courant peut être
        for j in range(num_tags): # compléter cette ligne

            # Initialiser best_prob pour le mot i à l'infini négatif
            best_prob_i = float("-inf")

            # Initialiser best_path_i le meilleur chemin pour le mot courant i à Aucun( None)
            best_path_i = None

            # Pour chaque POS tag que le mot précédent peut être :
            for k in range(num_tags):

                # Calculer la probabilité =
                # best probs du POS tag k, mot précédent i-1 +
                # log(prob de transition du POS k au POS j) +
                # log(prob que l'émission de POS j soit le mot i)
                prob = best_probs[k,i-1]+math.log(A[k,j]) +math.log(B[j,vocab[test_corpus[i]]])

                # vérifier si la probabilité de ce chemin est supérieure à
                # best_prob jusqu'à et avant ce point
                if prob > best_prob_i :

                    # Suivre la meilleure probabilité
                    best_prob_i = prob

                    # garder la trace du POS tag du mot précédent
                    # qui fait partie de best path.
                    # Enregistrez l'index (entier) associé au
                    # mot précédent le POS tag
                    best_path_i = k

            # Conserver la meilleure probabilité pour le
            # du POS tag du mot courant donnée
            # et la position du mot courant dans le corpus
            best_probs[j,i] = best_prob_i

            # Sauvegarder l'identifiant unique du POS tag précédent
            # dans la matrice best_paths, pour le POS tag du mot courant
            # et la position du mot courant dans le corpus.
            best_paths[j,i] = best_path_i


    return best_probs, best_paths

Exécutons la fonction `viterbi_forward` pour remplir les matrices `best_probs` et `best_paths`.

**Notons que cela prendra quelques minutes.  Il y a environ 30.000 mots à traiter**.

In [ ]:
# this will take a few minutes to run => processes ~ 30,000 words
best_probs, best_paths = viterbi_forward(A, B, prep, best_probs, best_paths, vocab)

Mots traités :     5000
Mots traités :    10000
Mots traités :    15000
Mots traités :    20000
Mots traités :    25000
Mots traités :    30000


In [ ]:
# Test this function
print(f"best_probs[0,1]: {best_probs[0,1]:.4f}")
print(f"best_probs[0,4]: {best_probs[0,4]:.4f}")

best_probs[0,1]: -24.7822
best_probs[0,4]: -49.5601


## <a name="vb">Viterbi backward</a>

Nous allons voir comment nous pouvons utiliser cette matrice de probabilité D que nous venons de créer pour reconstruire le chemin, afin de pouvoir identifier chaque mot avec le POS tag à attribuer.

La passe en arrière (backward) est la dernière des trois étapes de l'algorithme de Viterbi, où vous récupérerez la séquence la plus probable de parties de balises vocales pour votre séquence de mots donnée.

Grace au formules de la partie précésente nous avons maintenant rempli les matrices C et D. Il ne nous reste plus qu'à extraire le chemin à travers notre graphe de la matrice D, qui représente la séquence d'états cachés qui a le plus probablement généré notre séquence, du mot 1 jusqu'au mot K. Tout d'abord, calculons l'indice de l'entrée $c_{i,K}$ avec la plus grande probabilité dans la dernière colonne de C. La probabilité à cet indice est la probabilité de la séquence d'états cachés la plus probable, générant la séquence de mots donnée. Nous utilisons cet index s pour parcourir à l'envers la matrice D, afin de reconstituer la séquence de parties de POS Tag.

Tout d'abord, calculez l'indice de l'entrée $c_{i,K}$ avec la probabilité la plus élevée dans la dernière colonne de C. La probabilité à cet indice est la probabilité de la séquence la plus probable d'états cachés, générant la séquence de mots donnée.

Nous utilisons cet indice s pour parcourir à l'envers la matrice D, afin de reconstituer la séquence de POS tag.
<img src="images/bp1.png" style="width:700px;height:400;">

Voyons un exemple simple pour une matrice D, pour un modèle à quatre états et une séquence de mots donnée de longueur cinq. La matrice D, stocke toutes les étiquettes des états cachés que vous avez parcourus dans le cheminement vers l'avant.


Si nous revenons sur les états, en commençant par le chemin qui a la plus grande probabilité, nous obtenons effectivement la séquence la plus probable d'états cachés, ou de POS tag. Nous commencons par rechercher l'entrée ayant la plus forte probabilité dans la dernière ligne de la matrice C, et nous en extrayons l'indice s de cette entrée.
<img src="images/bp20.png" style="width:700px;height:400;">


Examinons la matrice C avec quelques probabilités que vous avez peut-être calculées lors de la passe en avant. Nous voulons maintenant calculer l'indice s de l'entrée dans la dernière ligne de C, qui a la plus forte probabilité.


L'entrée ayant la plus forte probabilité est la première avec 0,01. Ecrit sous forme de formule, **s est l'argmax de $c_{i,K}$**, qui dans ce cas est égal à 1.
<img src="images/bp2.png" style="width:700px;height:400;">


Cet indice représente le dernier état caché que vous avez traversé lorsque vous avez observé le mot $w_5$. Ainsi, les états les plus probables du mot $w_5$ sont sulement le POS tag $t_1$.
<img src="images/bp3.png" style="width:700px;height:400;">


Vous ajoutez donc $t_1$ à la fin de la séquence et recherchez l'index suivant dans D, qui vous indique d'où vous venez.

Cet index suivant est 3.

<img src="images/bp4.png" style="width:700px;height:400;">

Passez maintenant au quatrième mot de la séquence, ce qui signifie que nous regardons maintenant la quatrième colonne de la matrice D. Pour décider de la ligne de la matrice à examiner, rappelons-nous que dans la matrice D, colonne cinq, la ligne supérieure contient la valeur 3, ou 3 représente l'état précédent avec la plus grande probabilité. Donc $t_3$ est l'état le plus probable pour le mot numéro 4. Nous associons donc le mot numéro 4 à l'état numéro 3, qui est l'étiquette des parties du discours numéro 3.
<img src="images/bp5.png" style="width:700px;height:400;">
Nous pouvons continuer à marcher vers la gauche jusqu'à chaque colonne de la matrice D. Puisque la valeur stockée dans la colonne 4, ligne 3, est le nombre 1, nous pouvons attribuer le POS tag $t_1$ au mot précédent, le mot numéro 3.
<img src="images/bp5a.png" style="width:700px;height:400;">
De même, lorsque nous allons à gauche de la colonne 3, vous recherchez la ligne 1, qui représente l'état 1, que nous voyons surligné en vert. La valeur stockée dans la colonne 3, ligne 1 est 3, ce qui signifie que le mot précédent est associé au POS tag numéro 3. Donc, le mot numéro 2 est associé au POS tag 3.
<img src="images/bp6.png" style="width:700px;height:400;">
Maintenant, marchons vers la gauche d'une colonne dans la matrice jusqu'à la colonne 2. Puisque la valeur stockée dans la cellule verte de la colonne 3 est 3, allons à la ligne 3 de la colonne 2. Dans la colonne 2, la valeur stockée dans la ligne 3 est 2.
<img src="images/bp7.png" style="width:700px;height:400;">

Cela signifie que pour le mot 1, l'état le plus probable est le POS tag numéro 2. Et l'algorithme s'arrête car nous sommes arrivé au token de départ, les valeurs stockées dans la deuxième ligne de la première colonne étant 0. La séquence de ti que nous avons récupérée lors du "passage en arrière", est la séquence de POS tag avec la plus grande probabilité.
<img src="images/bp.png" style="width:700px;height:400;">
Avant de poursuivre, il convient d'être conscient de deux problèmes spécifiques à la mise en œuvre.
Lorsque vous implémentez l'algorithme de Viterbi dans le devoir de programmation, faites attention aux indices, car les listes d'indices de matrice en Python commencent par 0 au lieu de 1. Un autre problème spécifique à l'implémentation est que lorsque nous multiplions de nombreux nombres très petits comme des probabilités, cela entraîne des problèmes numériques, nous devons donc utiliser des probabilités logarithmiques à la place, où les nombres sont additionnés au lieu d'être multipliés.
<img src="images/bp02.png" style="width:700px;height:400;">
Ne vous inquiétez pas, je reviendrai sur ce concept plus tard dans le cours. Félicitations pour avoir terminé cette semaine. Nous connaissons maintenant l'algorithme de Viterbi, et nous l'avons notamment utilisé pour le marquage des POS tag. Nous pouvons utiliser **l'étiquetage morpho-syntaxique** (POS tag) pour la recherche, la traduction automatique, la reconnaissance vocale et l'analyse syntaxique.

Retour sur nos donnée....La partie backward de l'algorithme de Viterbi obtient les prédictions des POS tag pour chaque mot du corpus en utilisant les matrices `best_paths` et `best_probs`.

Implémentons l'algorithme `viterbi_backward`, qui retourne une liste de POS tag pédites pour chaque mot du corpus.

- Notons que la numérotation des positions de l'index commence à 0 et non à 1.
- m' est le nombre de mots dans le corpus.  
- Ainsi, l'indexation dans le corpus va de `0` à `m - 1`.
- De même, les colonnes dans `best_probs` et `best_paths` sont indexées de `0` à `m - 1`.


**À l'étape 1:**       
Passez en boucle toutes les lignes (POS tag) dans la dernière entrée de `best_probs` et trouvez la ligne (POS tag) avec la valeur maximale.
Convertissez l'ID unique de l'entier en un tag (une représentation de chaîne de caractères) en utilisant le dictionnaire `states`.  



**À l'étape 2:**  
- En commençant à la dernière colonne de best_paths, utilisons `best_probs` pour trouver le POS Tag la plus probable pour le dernier mot du corpus.
- Ensuite, utilisons les `best_paths` pour trouver le POS tag le plus probable pour le mot précédent.
- Mettez à jour le POS tag pour chaque mot dans `z` et dans `preds` où La variable `z` est un tableau qui stocke l'ID: entier unique (l'indice comme **cindex**) des POS Tag prédit pour chaque mot du corpus.



In [ ]:
def viterbi_backward(best_probs, best_paths, corpus, states) :
    '''
    Cette fonction renvoie le meilleur chemin.

    '''
    # Obtenir le nombre de mots dans le corpus
    # qui est aussi le nombre de colonnes dans best_probs, best_paths
    m = best_paths.shape[1]

    # Initialiser le tableau z, de même longueur que le corpus
    z = [None] * m

    # Obtenir le nombre de Pos tag unique
    num_tags = best_probs.shape[0]

    # Initialiser la meilleure probabilité pour le dernier mot
    best_prob_for_last_word = float('-inf')

    # Initialise le tableau de prédonnées, de même longueur que le corpus
    pred = [None] * m


    ## Étape 1 ##

    # Passez en revue chaque POS tag pour le dernier mot (dernière colonne de best_probs)
    # afin de trouver la ligne (ID d'entier du tag POS)
    # avec la plus grande probabilité pour le dernier mot
    for k in range(num_tags): # compléter cette ligne

        # Si la probabilité d'un POS tag à la ligne k
        # est meilleur que la meilleure probabilité précédente pour le dernier mot :
        if best_probs[k,-1]>best_prob_for_last_word: # complétez cette ligne

            # Stocker la nouvelle meilleure probabilité pour le mot lsat
            best_prob_for_last_word = best_probs [k,-1]

            # Stocker l'identifiant unique du POS tag
            # qui est aussi le numéro de ligne dans best_probs
            z[m - 1] = k

    # Convertir le POS tag prédit du dernier mot
    # de son identifiant unique d'entier dans la représentation de la chaîne
    # en utilisant le dictionnaire des "états
    # enregistrer ceci dans le tableau "pred" pour le dernier mot
    pred[m - 1] = states[k]

    ## Étape 2 ##
    # Trouver les meilleures POS tag en marchant à reculons dans best_paths
    # Du dernier mot du corpus au 0ème mot du corpus  0 car dans python les indices comme à zéro
    for i in range(len(corpus)-1, -1, -1): # compléter cette ligne

        # Récupérez l'identifiant unique de l'entier
        # le POS tag pour le mot à la position "i" dans le corpus
        pos_tag_for_word_i = best_paths[np.argmax(best_probs[ :,i]),i]

        # Dans best_paths, allez à la ligne représentant le POS tag du mot i
        # et la colonne représentant la position du mot dans le corpus
        # pour récupérer le POS prévu pour le mot en position i-1 dans le corpus
        z[i - 1] = best_paths[pos_tag_for_word_i,i]

        # Obtenir le POS tag du mot précédent sous forme de chaîne
        # Utilisez le dictionnaire des "états",
        # où la clé est l'ID: entier unique du Pos tag,
        # et la valeur est la représentation de la chaîne de ce Pos tag
        pred[i - 1] = states[pos_tag_for_word_i]

    return pred

In [ ]:
# test
pred = viterbi_backward(best_probs, best_paths, prep, states)
m=len(pred)
print('La prediction pour pred[-7:m-1] est : \n', prep[-7:m-1], "\n", pred[-7:m-1], "\n")
print('La prediction pour pred[0:8] est: \n', pred[0:7], "\n", prep[0:7])

La prediction pour pred[-7:m-1] est : 
 ['see', 'them', 'here', 'with', 'us', '.'] 
 ['VB', 'PRP', 'RB', 'IN', 'PRP', '.'] 

La prediction pour pred[0:8] est: 
 ['DT', 'NN', 'POS', 'NN', 'MD', 'VB', 'VBN'] 
 ['The', 'economy', "'s", 'temperature', 'will', 'be', 'taken']


In [ ]:
print('Le troisième mot est:', prep[3])
print('Notre prediction est:', pred[3])
print('Notre label y corepondant est: ', y[3])

Le troisième mot est: temperature
Notre prediction est: NN
Notre label y corepondant est:  temperature	NN



Mettons en œuvre une fonction permettant de calculer la précision des prédictions de l'algorithme de Viterbi pour les POS tags.
- Pour diviser y en le mot et son étiquette nous pouvons utiliser `y.split()`.

In [ ]:
def compute_accuracy(pred, y) :
    '''
    Entrée :
        pred : une liste des POS prédites
        y : une liste de lignes où chaque mot est séparé par un "\t" (i.e. mot \t tag)
    Sortie :

    '''
    num_correct = 0
    total = 0

    # Réunir la prédiction et les étiquettes
    for prediction, y in zip(pred, y):

        # Séparer l'étiquette en deux parties : le mot et le POS tag
        word_tag_tuple = y.split()

        # Vérifier qu'il y a bien un mot et un tag
        # pas plus et pas moins de 2 articles
        if len(word_tag_tuple)!=2 :
            continue

        # stocker le mot et l'étiquette ( Pos tag) séparément
        word, tag = word_tag_tuple

        # Vérifier si le POS tag correspond à la prédiction
        if prediction == tag : # compléter cette ligne

            # compter le nombre de fois que la prédiction
            # correspond à l'étiquette
            num_correct += 1

        # garder une trace du nombre total d'exemples (qui ont des étiquettes valides)
        total += 1

    return num_correct/total

In [ ]:
print(f"Précision de l'algorithme de Viterbi: {compute_accuracy(pred, y):.4f}")

Précision de l'algorithme de Viterbi: 0.9528


# <a name="5">V. Référence </a>

- [Coursera Probabilistic models in nlp](https://www.coursera.org/learn/probabilistic-models-in-nlp#syllabus)